# RQ3 Colab runner: time-aware TIGER

This notebook clones the GitHub repository into Google Colab and runs Research Question 3: whether explicit modeling of time gaps improves generative retrieval.

The main comparison is on the temporal split: plain TIGER, TIGER with discrete gap tokens, and TIGER with a learned gap embedding added to event token embeddings. A leave-one-out TIGER run is included only as the reproduction reference.

In [ ]:
REPO_URL = "https://github.com/MayaLager/TIGER-modi.git"
BRANCH = "ksusha/errors-fix"
PROJECT_DIR = "/content/TIGER-modi"

USE_DRIVE_CACHE = True
DRIVE_ROOT = "/content/drive/MyDrive/TIGER-modi-colab"

CATEGORY = "Beauty"
CODEBOOK_SIZE = 256
RQVAE_EPOCHS = 500
TIGER_EPOCHS = 100
LR = "3e-4"
MAX_ITEMS = 20
NUM_BEAMS = 100
SEED = 42
DEVICE = "cuda"
FORCE_RERUN = False

DATA_DIR = f"data/processed/{CATEGORY}"
TEMP_DATA_DIR = f"data/processed/{CATEGORY}_temporal"
CODES = f"{DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}.npy"
TEMP_CODES = f"{TEMP_DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}_fit-temporal.npy"
RUN_DIR = "runs/rq3"
LOG_DIR = f"{RUN_DIR}/logs"
METRICS_DIR = f"{RUN_DIR}/metrics"

In [ ]:
!nvidia-smi || true
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not os.path.exists(PROJECT_DIR):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}")
else:
    sh("git fetch origin", cwd=PROJECT_DIR)
    sh(f"git checkout {BRANCH}", cwd=PROJECT_DIR)
    sh("git pull --ff-only", cwd=PROJECT_DIR)

%cd {PROJECT_DIR}
!git status --short --branch
!git log --oneline -3

In [ ]:
import pathlib
import shutil

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = pathlib.Path(DRIVE_ROOT)
    for sub in ["data/raw", "data/processed", "runs"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    pathlib.Path("data").mkdir(exist_ok=True)
    for local, target in [
        ("data/raw", root / "data/raw"),
        ("data/processed", root / "data/processed"),
        ("runs", root / "runs"),
    ]:
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(target, local)

for path in [RUN_DIR, LOG_DIR, METRICS_DIR]:
    os.makedirs(path, exist_ok=True)

!ls -lah data runs

In [ ]:
!python -m pip install -q -r requirements.txt
!python - <<'PY'
import torch, transformers, sklearn
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
PY

In [ ]:
import json
import time

def run_logged(args, out_path=None, log_path=None, force=False):
    if out_path and os.path.exists(out_path) and not force:
        print(f"[skip] {out_path} exists")
        return
    if out_path:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
    if log_path:
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
    print("$ " + " ".join(map(str, args)))
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    with open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w") as log:
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    rc = proc.wait()
    print(f"[exit {rc}] elapsed={(time.time() - t0) / 60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")

def run_json(name):
    return f"{RUN_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.json"

def run_log(name):
    return f"{LOG_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.log"

def metrics_log(name):
    return f"{METRICS_DIR}/{CATEGORY}_{name}_lr3e4_e100_seed{SEED}.jsonl"

## Data and Semantic IDs

In [ ]:
run_logged(["python", "src/data/download.py", "--category", CATEGORY, "--out", "data/raw", "--review-set", "5core"], log_path=f"{LOG_DIR}/download_5core.log", force=True)
run_logged(["python", "src/data/download.py", "--category", CATEGORY, "--out", "data/raw", "--review-set", "all"], log_path=f"{LOG_DIR}/download_all.log", force=True)

run_logged(["python", "src/data/preprocess.py", "--category", CATEGORY, "--raw", "data/raw", "--out", DATA_DIR, "--kcore", "5"], out_path=f"{DATA_DIR}/meta.json", log_path=f"{LOG_DIR}/preprocess_leave_one_out.log", force=FORCE_RERUN)
run_logged(["python", "src/data/preprocess.py", "--category", CATEGORY, "--raw", "data/raw", "--out", TEMP_DATA_DIR, "--kcore", "5", "--kcore-scope", "train_temporal", "--review-set", "all"], out_path=f"{TEMP_DATA_DIR}/meta.json", log_path=f"{LOG_DIR}/preprocess_temporal.log", force=FORCE_RERUN)

print(json.dumps(json.load(open(f"{DATA_DIR}/meta.json")), indent=2))
print(json.dumps(json.load(open(f"{TEMP_DATA_DIR}/meta.json")), indent=2))

In [ ]:
run_logged([
    "python", "src/train_semantic_ids.py",
    "--data", DATA_DIR,
    "--id-method", "rqvae",
    "--num-levels", "3",
    "--codebook-size", str(CODEBOOK_SIZE),
    "--rqvae-epochs", str(RQVAE_EPOCHS),
    "--device", DEVICE,
    "--seed", str(SEED),
], out_path=f"{DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}.meta.json", log_path=f"{LOG_DIR}/semantic_ids_leave_one_out.log", force=FORCE_RERUN)

run_logged([
    "python", "src/train_semantic_ids.py",
    "--data", TEMP_DATA_DIR,
    "--id-method", "rqvae",
    "--num-levels", "3",
    "--codebook-size", str(CODEBOOK_SIZE),
    "--rqvae-epochs", str(RQVAE_EPOCHS),
    "--fit-split", "temporal",
    "--device", DEVICE,
    "--seed", str(SEED),
], out_path=f"{TEMP_DATA_DIR}/codes_rqvae_L3_K{CODEBOOK_SIZE}_fit-temporal.meta.json", log_path=f"{LOG_DIR}/semantic_ids_temporal.log", force=FORCE_RERUN)

!ls -lh data/processed/Beauty/codes_rqvae_L3_K256.npy data/processed/Beauty_temporal/codes_rqvae_L3_K256_fit-temporal.npy

## RQ3 Runs

In [ ]:
def tiger_args(data_dir, codes, split, name, extra=None):
    args = [
        "python", "-u", "src/train_tiger.py",
        "--data", data_dir,
        "--codes", codes,
        "--codebook-size", str(CODEBOOK_SIZE),
        "--max-items", str(MAX_ITEMS),
        "--split", split,
        "--epochs", str(TIGER_EPOCHS),
        "--lr", LR,
        "--num-beams", str(NUM_BEAMS),
        "--device", DEVICE,
        "--seed", str(SEED),
        "--metrics-log", metrics_log(name),
        "--out", run_json(name),
    ]
    if extra:
        args.extend(extra)
    return args

def run_tiger(name, data_dir, codes, split, extra=None):
    run_logged(tiger_args(data_dir, codes, split, name, extra), out_path=run_json(name), log_path=run_log(name), force=FORCE_RERUN)

In [ ]:
run_tiger("tiger_leave_one_out", DATA_DIR, CODES, "leave_one_out")

In [ ]:
run_tiger("tiger_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal")

In [ ]:
run_tiger("tiger_time_gaps_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal", ["--time-gaps"])

In [ ]:
run_tiger("tiger_time_embed_temporal", TEMP_DATA_DIR, TEMP_CODES, "temporal", ["--time-embed"])

## Summary

In [ ]:
import glob

rows = []
for path in sorted(glob.glob(f"{RUN_DIR}/{CATEGORY}_*.json")):
    d = json.load(open(path))
    cfg = d.get("config", {})
    test = d.get("test", {})
    diag = d.get("split_diagnostics", {})
    rows.append({
        "run": os.path.basename(path),
        "split": cfg.get("split"),
        "time_gaps": cfg.get("time_gaps"),
        "time_embed": cfg.get("time_embed"),
        "best_val_R@10": d.get("best_val_recall@10"),
        "test_R@5": test.get("recall@5"),
        "test_N@5": test.get("ndcg@5"),
        "test_R@10": test.get("recall@10"),
        "test_N@10": test.get("ndcg@10"),
        "invalid_code_rate": test.get("invalid_code_rate"),
        "decode_latency_s": test.get("decode_latency_s"),
        "test_seen_target_rate": (diag.get("test_target_coverage") or {}).get("seen_target_rate"),
        "path": path,
    })

summary_csv = f"{RUN_DIR}/summary_rq3_lr3e4_e100_seed{SEED}.csv"
summary_md = f"{RUN_DIR}/summary_rq3_lr3e4_e100_seed{SEED}.md"

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
    df.to_csv(summary_csv, index=False)
    md = df.to_markdown(index=False)
except Exception:
    md = "\n".join(str(r) for r in rows)

with open(summary_md, "w", encoding="utf-8") as f:
    f.write(md + "\n")

print("saved", summary_csv)
print("saved", summary_md)
print(md)